# Suraksha - Synthetic UPI Transaction Data Generator

**Generates 500,000 UPI transactions with 9 fraud types and 35+ features**

Person 1's First Task - Start immediately at 6:30 PM

## Fraud Distribution:
- **Total Transactions**: 500,000 (increased for realism)
- **Legitimate**: 475,000 (95%)
- **Fraud**: 25,000 (5%)
  - Velocity Fraud: 3,750 (15%)
  - Mule Account: 3,250 (13%)
  - SIM Swap: 3,000 (12%)
  - Device Takeover: 3,000 (12%)
  - Beneficiary Manipulation: 2,500 (10%)
  - Amount Anomaly: 3,250 (13%)
  - Temporal Anomaly: 3,000 (12%)
  - Merchant Fraud: 2,000 (8%)
  - Failed-Then-Success: 1,250 (5%)

## Why 500K Transactions?
- **Realistic fraud patterns**: Each fraud type has sufficient examples (1,250-3,750 cases)
- **Avoid 100% precision**: Larger dataset prevents overfitting on small classes
- **Production-scale**: Represents ~1.5 days of data for medium-sized bank
- **Better model generalization**: More diverse patterns and edge cases

## Key Features:
- ✅ User identifiers (VPAs, device IDs, SIM metadata)
- ✅ Behavioral tracking (velocity, transaction history)
- ✅ Temporal patterns (date, time, day of week)
- ✅ Geographic data (sender/receiver states)
- ✅ Device & security metadata (MPIN attempts, device changes)
- ✅ 9 distinct fraud types with realistic signatures

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid
import hashlib
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configuration - INCREASED FOR REALISM
TOTAL_TRANSACTIONS = 500000  # Increased from 120K to 500K for realistic fraud patterns
FRAUD_RATIO = 0.05
NUM_USERS = 25000  # Increased proportionally from 10K
NUM_MERCHANTS = 5000  # Increased proportionally from 2K

# Fraud type distribution (within 5% fraud transactions)
FRAUD_DISTRIBUTION = {
    'Velocity Fraud': 0.15,
    'Mule Account': 0.13,
    'SIM Swap': 0.12,
    'Device Takeover': 0.12,
    'Beneficiary Manipulation': 0.10,
    'Amount Anomaly': 0.13,
    'Temporal Anomaly': 0.12,
    'Merchant Fraud': 0.08,
    'Failed-Then-Success': 0.05
}

# Constants
BANKS = ['HDFC', 'ICICI', 'SBI', 'Axis', 'Kotak', 'YES Bank', 'Paytm', 'PhonePe', 'GooglePay']
STATES = ['Maharashtra', 'Delhi', 'Karnataka', 'Tamil Nadu', 'Gujarat', 'Uttar Pradesh', 
          'West Bengal', 'Rajasthan', 'Kerala', 'Telangana']
AGE_GROUPS = ['18-25', '26-35', '36-45', '46-55', '56+']
DEVICE_TYPES = ['Android', 'iOS', 'Web']
NETWORK_TYPES = ['4G', '5G', 'WiFi', '3G']
TXN_TYPES = ['P2P', 'P2M', 'Bill Payment', 'Recharge']
MERCHANT_CATEGORIES = ['Food', 'Shopping', 'Fuel', 'Entertainment', 'Travel', 'Healthcare', 
                       'Education', 'Utilities', 'Electronics', 'Jewelry', 'Gambling', 'Adult']

print(f"✅ Configuration loaded")
print(f"Target: {TOTAL_TRANSACTIONS:,} transactions (INCREASED FOR REALISM)")
print(f"Fraud ratio: {FRAUD_RATIO*100}%")
print(f"Total fraud cases: {int(TOTAL_TRANSACTIONS * FRAUD_RATIO):,}")
print(f"Users: {NUM_USERS:,}")
print(f"Merchants: {NUM_MERCHANTS:,}")

In [0]:
def generate_vpa(user_id, bank):
    """Generate VPA like alice@paytm"""
    names = ['alice', 'bob', 'charlie', 'dave', 'eve', 'frank', 'grace', 'henry', 
             'ivy', 'jack', 'kate', 'leo', 'mia', 'noah', 'olivia', 'peter']
    base_name = random.choice(names)
    return f"{base_name}{user_id % 1000}@{bank.lower()}"

def hash_id(value):
    """Create hashed user ID"""
    return hashlib.md5(str(value).encode()).hexdigest()[:16]

print("✅ Helper functions defined")

In [0]:
print("📊 Generating user profiles...")
users = []
for i in range(NUM_USERS):
    users.append({
        'user_id': hash_id(i),
        'bank': random.choice(BANKS),
        'state': random.choice(STATES),
        'age_group': random.choice(AGE_GROUPS),
        'device_id': f"DEV_{hash_id(i)[:8]}",
        'device_type': random.choice(DEVICE_TYPES),
        'sim_age_days': random.randint(30, 1000),
        'avg_txn_amount': random.uniform(500, 5000)
    })
user_df = pd.DataFrame(users)
print(f"✓ Created {len(users):,} user profiles")
user_df.head()

In [0]:
print("📊 Generating merchant profiles...")
merchants = []
for i in range(NUM_MERCHANTS):
    merchants.append({
        'merchant_id': hash_id(f"merchant_{i}"),
        'merchant_vpa': f"merchant{i}@{random.choice(BANKS).lower()}",
        'category': random.choice(MERCHANT_CATEGORIES),
        'is_high_risk': random.random() < 0.1  # 10% high risk
    })
merchant_df = pd.DataFrame(merchants)
print(f"✓ Created {len(merchants):,} merchant profiles")
merchant_df.head()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("⚪ Generating legitimate transactions using PySpark...")

# Convert pandas DataFrames to Spark DataFrames for joining
user_spark_df = spark.createDataFrame(user_df)
merchant_spark_df = spark.createDataFrame(merchant_df)

# Create a range DataFrame with transaction IDs
legit_txns = spark.range(legitimate_count).select(
    F.col("id").alias("txn_seq"),
    F.concat(F.lit("TXN_"), F.substring(F.md5(F.concat(F.lit("legit_"), F.col("id").cast("string"))), 1, 12)).alias("txn_id")
)

# Add random user assignment
legit_txns = legit_txns.withColumn(
    "user_idx", (F.rand(42) * NUM_USERS).cast("int")
)

# Join with user data
user_indexed = user_spark_df.withColumn("user_idx", F.monotonically_increasing_id() % NUM_USERS)
legit_txns = legit_txns.join(F.broadcast(user_indexed), "user_idx", "left")

# Add timestamp - proper approach using unix_timestamp
base_timestamp = int(start_date.timestamp())
legit_txns = legit_txns.withColumn(
    "timestamp",
    F.from_unixtime(
        F.lit(base_timestamp) + 
        (F.rand() * date_range * 86400).cast("long") +  # Random days in seconds
        ((F.rand() * 17 + 6).cast("int") * 3600) +  # Hours 6-22 in seconds
        ((F.rand() * 60).cast("int") * 60)  # Random minutes in seconds
    ).cast("timestamp")
)

# Add transaction type
legit_txns = legit_txns.withColumn(
    "txn_type", 
    F.element_at(F.array([F.lit(t) for t in TXN_TYPES]), (F.rand() * len(TXN_TYPES) + 1).cast("int"))
)

legit_txns = legit_txns.withColumn(
    "sender_vpa", F.concat(
        F.element_at(F.array([F.lit(n) for n in ['alice', 'bob', 'charlie', 'dave', 'eve', 'frank', 'grace', 'henry', 'ivy', 'jack']]), 
                     (F.rand() * 10 + 1).cast("int")),
        (F.hash(F.col("user_id")) % 1000).cast("string"),
        F.lit("@"),
        F.lower(F.col("bank"))
    )
)

legit_txns = legit_txns \
    .withColumn("sender_id", F.col("user_id")) \
    .withColumn("sender_bank", F.col("bank")) \
    .withColumn("sender_state", F.col("state")) \
    .withColumn("sender_age_group", F.col("age_group")) \
    .withColumn("network_type", F.element_at(F.array([F.lit(n) for n in NETWORK_TYPES]), (F.rand() * len(NETWORK_TYPES) + 1).cast("int"))) \
    .withColumn("amount_inr", F.round(F.exp(F.log(F.col("avg_txn_amount")) + F.randn() * 0.5), 2)) \
    .withColumn("txn_status", F.lit("SUCCESS")) \
    .withColumn("mpin_attempts", F.lit(0)) \
    .withColumn("device_changed_flag", F.when(F.rand(123) < 0.12, True).otherwise(False)) \
    .withColumn("sim_change_recent", F.when(F.rand(456) < 0.05, True).otherwise(False)) \
    .withColumn("location_changed_flag", F.when(F.rand(789) < 0.10, True).otherwise(False)) \
    .withColumn("is_fraud", F.lit(0)) \
    .withColumn("fraud_type", F.lit("Legitimate")) \
    .withColumn("fraud_type_id", F.lit(0))

# Split into P2M and P2P transactions
p2m_txns = legit_txns.filter(F.col("txn_type") == "P2M") \
    .withColumn("merchant_idx", (F.rand() * NUM_MERCHANTS).cast("int"))

merchant_indexed = merchant_spark_df.withColumn("merchant_idx", F.monotonically_increasing_id() % NUM_MERCHANTS)
p2m_txns = p2m_txns.join(F.broadcast(merchant_indexed), "merchant_idx", "left") \
    .withColumn("receiver_id", F.col("merchant_id")) \
    .withColumn("receiver_vpa", F.col("merchant_vpa")) \
    .withColumn("receiver_bank", F.element_at(F.array([F.lit(b) for b in BANKS]), (F.rand() * len(BANKS) + 1).cast("int"))) \
    .withColumn("receiver_state", F.element_at(F.array([F.lit(s) for s in STATES]), (F.rand() * len(STATES) + 1).cast("int"))) \
    .withColumn("receiver_age_group", F.lit("NA")) \
    .withColumn("merchant_category", F.col("category")) \
    .withColumn("high_risk_merchant", F.col("is_high_risk"))

p2p_txns = legit_txns.filter(F.col("txn_type") != "P2M") \
    .withColumn("receiver_idx", (F.rand(99) * NUM_USERS).cast("int"))

receiver_indexed = user_spark_df.withColumn("receiver_idx", F.monotonically_increasing_id() % NUM_USERS) \
    .select(
        F.col("receiver_idx"),
        F.col("user_id").alias("receiver_id"),
        F.col("bank").alias("receiver_bank"),
        F.col("state").alias("receiver_state"),
        F.col("age_group").alias("receiver_age_group")
    )

p2p_txns = p2p_txns.join(F.broadcast(receiver_indexed), "receiver_idx", "left") \
    .withColumn(
        "receiver_vpa", F.concat(
            F.element_at(F.array([F.lit(n) for n in ['alice', 'bob', 'charlie', 'dave', 'eve']]), 
                         (F.rand(88) * 5 + 1).cast("int")),
            (F.hash(F.col("receiver_id")) % 1000).cast("string"),
            F.lit("@"),
            F.lower(F.col("receiver_bank"))
        )
    ) \
    .withColumn("merchant_category", F.lit(None).cast("string")) \
    .withColumn("high_risk_merchant", F.lit(False))

# Union P2M and P2P
legit_final = p2m_txns.unionByName(p2p_txns, allowMissingColumns=True)

# Select final columns
final_columns = ['txn_id', 'timestamp', 'sender_id', 'sender_vpa', 'sender_bank', 'sender_state', 
                 'sender_age_group', 'device_id', 'device_type', 'network_type', 'txn_type', 
                 'amount_inr', 'txn_status', 'mpin_attempts', 'device_changed_flag', 'sim_change_recent', 
                 'sim_age_days', 'location_changed_flag', 'receiver_id', 'receiver_vpa', 'receiver_bank',
                 'receiver_state', 'receiver_age_group', 'merchant_category', 'high_risk_merchant',
                 'is_fraud', 'fraud_type', 'fraud_type_id']

legit_final = legit_final.select(*final_columns)

print(f"✓ Generated {legit_final.count():,} legitimate transactions using Spark")
print("✓ Added realistic noise: ~12% device changes, ~5% SIM changes, ~10% location changes")
legit_final.show(5)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("⚪ Generating legitimate transactions using PySpark...")

# Convert pandas DataFrames to Spark DataFrames for joining
user_spark_df = spark.createDataFrame(user_df)
merchant_spark_df = spark.createDataFrame(merchant_df)

# Create a range DataFrame with transaction IDs
legit_txns = spark.range(legitimate_count).select(
    F.col("id").alias("txn_seq"),
    F.concat(F.lit("TXN_"), F.substring(F.md5(F.concat(F.lit("legit_"), F.col("id").cast("string"))), 1, 12)).alias("txn_id")
)

# Add random user assignment
legit_txns = legit_txns.withColumn(
    "user_idx", (F.rand(42) * NUM_USERS).cast("int")
)

# Join with user data
user_indexed = user_spark_df.withColumn("user_idx", F.monotonically_increasing_id() % NUM_USERS)
legit_txns = legit_txns.join(F.broadcast(user_indexed), "user_idx", "left")

# Add timestamp - proper approach using unix_timestamp
base_timestamp = int(start_date.timestamp())
legit_txns = legit_txns.withColumn(
    "timestamp",
    F.from_unixtime(
        F.lit(base_timestamp) + 
        (F.rand() * date_range * 86400).cast("long") +  # Random days in seconds
        ((F.rand() * 17 + 6).cast("int") * 3600) +  # Hours 6-22 in seconds
        ((F.rand() * 60).cast("int") * 60)  # Random minutes in seconds
    ).cast("timestamp")
)

# Add transaction type
legit_txns = legit_txns.withColumn(
    "txn_type", 
    F.element_at(F.array([F.lit(t) for t in TXN_TYPES]), (F.rand() * len(TXN_TYPES) + 1).cast("int"))
)

legit_txns = legit_txns.withColumn(
    "sender_vpa", F.concat(
        F.element_at(F.array([F.lit(n) for n in ['alice', 'bob', 'charlie', 'dave', 'eve', 'frank', 'grace', 'henry', 'ivy', 'jack']]), 
                     (F.rand() * 10 + 1).cast("int")),
        (F.hash(F.col("user_id")) % 1000).cast("string"),
        F.lit("@"),
        F.lower(F.col("bank"))
    )
)

legit_txns = legit_txns \
    .withColumn("sender_id", F.col("user_id")) \
    .withColumn("sender_bank", F.col("bank")) \
    .withColumn("sender_state", F.col("state")) \
    .withColumn("sender_age_group", F.col("age_group")) \
    .withColumn("network_type", F.element_at(F.array([F.lit(n) for n in NETWORK_TYPES]), (F.rand() * len(NETWORK_TYPES) + 1).cast("int"))) \
    .withColumn("amount_inr", F.round(F.exp(F.log(F.col("avg_txn_amount")) + F.randn() * 0.5), 2)) \
    .withColumn("txn_status", F.lit("SUCCESS")) \
    .withColumn("mpin_attempts", F.lit(0)) \
    .withColumn("device_changed_flag", F.lit(False)) \
    .withColumn("sim_change_recent", F.lit(False)) \
    .withColumn("location_changed_flag", F.lit(False)) \
    .withColumn("is_fraud", F.lit(0)) \
    .withColumn("fraud_type", F.lit("Legitimate")) \
    .withColumn("fraud_type_id", F.lit(0))

# Split into P2M and P2P transactions
p2m_txns = legit_txns.filter(F.col("txn_type") == "P2M") \
    .withColumn("merchant_idx", (F.rand() * NUM_MERCHANTS).cast("int"))

merchant_indexed = merchant_spark_df.withColumn("merchant_idx", F.monotonically_increasing_id() % NUM_MERCHANTS)
p2m_txns = p2m_txns.join(F.broadcast(merchant_indexed), "merchant_idx", "left") \
    .withColumn("receiver_id", F.col("merchant_id")) \
    .withColumn("receiver_vpa", F.col("merchant_vpa")) \
    .withColumn("receiver_bank", F.element_at(F.array([F.lit(b) for b in BANKS]), (F.rand() * len(BANKS) + 1).cast("int"))) \
    .withColumn("receiver_state", F.element_at(F.array([F.lit(s) for s in STATES]), (F.rand() * len(STATES) + 1).cast("int"))) \
    .withColumn("receiver_age_group", F.lit("NA")) \
    .withColumn("merchant_category", F.col("category")) \
    .withColumn("high_risk_merchant", F.col("is_high_risk"))

p2p_txns = legit_txns.filter(F.col("txn_type") != "P2M") \
    .withColumn("receiver_idx", (F.rand(99) * NUM_USERS).cast("int"))

receiver_indexed = user_spark_df.withColumn("receiver_idx", F.monotonically_increasing_id() % NUM_USERS) \
    .select(
        F.col("receiver_idx"),
        F.col("user_id").alias("receiver_id"),
        F.col("bank").alias("receiver_bank"),
        F.col("state").alias("receiver_state"),
        F.col("age_group").alias("receiver_age_group")
    )

p2p_txns = p2p_txns.join(F.broadcast(receiver_indexed), "receiver_idx", "left") \
    .withColumn(
        "receiver_vpa", F.concat(
            F.element_at(F.array([F.lit(n) for n in ['alice', 'bob', 'charlie', 'dave', 'eve']]), 
                         (F.rand(88) * 5 + 1).cast("int")),
            (F.hash(F.col("receiver_id")) % 1000).cast("string"),
            F.lit("@"),
            F.lower(F.col("receiver_bank"))
        )
    ) \
    .withColumn("merchant_category", F.lit(None).cast("string")) \
    .withColumn("high_risk_merchant", F.lit(False))

# Union P2M and P2P
legit_final = p2m_txns.unionByName(p2p_txns, allowMissingColumns=True)

# Select final columns
final_columns = ['txn_id', 'timestamp', 'sender_id', 'sender_vpa', 'sender_bank', 'sender_state', 
                 'sender_age_group', 'device_id', 'device_type', 'network_type', 'txn_type', 
                 'amount_inr', 'txn_status', 'mpin_attempts', 'device_changed_flag', 'sim_change_recent', 
                 'sim_age_days', 'location_changed_flag', 'receiver_id', 'receiver_vpa', 'receiver_bank',
                 'receiver_state', 'receiver_age_group', 'merchant_category', 'high_risk_merchant',
                 'is_fraud', 'fraud_type', 'fraud_type_id']

legit_final = legit_final.select(*final_columns)

print(f"✓ Generated {legit_final.count():,} legitimate transactions using Spark")
legit_final.show(5)

In [0]:
print("\n🔴 Generating fraud transactions using PySpark (WITH REALISTIC NOISE)...\n")

fraud_dfs = []
fraud_type_id = 1
base_timestamp = int(start_date.timestamp())

for fraud_type, count in fraud_counts.items():
    print(f"  Generating {fraud_type}: {count:,} transactions...")
    
    # Create base fraud transactions
    fraud_txns = spark.range(count).select(
        F.col("id").alias("fraud_seq"),
        F.concat(F.lit("TXN_"), F.substring(F.md5(F.concat(F.lit(fraud_type), F.col("id").cast("string"))), 1, 12)).alias("txn_id")
    )
    
    # Add user assignment
    fraud_txns = fraud_txns.withColumn("user_idx", (F.rand(fraud_type_id * 100) * NUM_USERS).cast("int"))
    fraud_txns = fraud_txns.join(F.broadcast(user_indexed), "user_idx", "left")
    
    # Fraud-specific patterns - NOW WITH REALISTIC NOISE
    if fraud_type == 'Velocity Fraud':
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 24).cast("int") * 3600) + ((F.rand() * 60).cast("int") * 60)).cast("timestamp")) \
            .withColumn("amount_inr", F.round(F.lit(1000) + F.rand() * 9000, 2)) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", F.lit(0)) \
            .withColumn("location_changed_flag", F.lit(False))
    
    elif fraud_type == 'Mule Account':
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 24).cast("int") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.element_at(F.array([F.lit(5000.0), F.lit(10000.0), F.lit(15000.0), F.lit(20000.0), F.lit(25000.0)]), (F.rand() * 5 + 1).cast("int"))) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", F.lit(0)) \
            .withColumn("location_changed_flag", F.lit(False))
    
    elif fraud_type == 'SIM Swap':
        # NOISE: Only ~75% have sim_change_recent=True (was 100%)
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 24).cast("int") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.round(F.lit(5000) + F.rand() * 25000, 2)) \
            .withColumn("device_changed_flag", F.when(F.rand(1001) < 0.80, True).otherwise(False)) \
            .withColumn("sim_change_recent", F.when(F.rand(1002) < 0.75, True).otherwise(False)) \
            .withColumn("mpin_attempts", (F.rand() * 2 + 1).cast("int")) \
            .withColumn("location_changed_flag", F.lit(False)) \
            .withColumn("device_id", F.concat(F.lit("DEV_"), F.substring(F.md5(F.concat(F.lit("new_"), F.col("fraud_seq").cast("string"))), 1, 8))) \
            .withColumn("sim_age_days", (F.rand() * 7 + 1).cast("int"))
    
    elif fraud_type == 'Device Takeover':
        # NOISE: ~80% have device_changed_flag, ~80% have location_changed_flag (was 100% both)
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 24).cast("int") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.round(F.col("avg_txn_amount") * (F.lit(2.5) + F.rand() * 2.5), 2)) \
            .withColumn("device_changed_flag", F.when(F.rand(2001) < 0.80, True).otherwise(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", (F.rand() * 3 + 1).cast("int")) \
            .withColumn("location_changed_flag", F.when(F.rand(2002) < 0.80, True).otherwise(False)) \
            .withColumn("device_id", F.concat(F.lit("DEV_"), F.substring(F.md5(F.concat(F.lit("takeover_"), F.col("fraud_seq").cast("string"))), 1, 8)))
    
    elif fraud_type == 'Beneficiary Manipulation':
        # NOISE: ~70% at unusual hours [0-4,23], ~30% at normal hours [6-22]
        fraud_txns = fraud_txns \
            .withColumn("is_unusual", F.rand(3001) < 0.70) \
            .withColumn("hour", 
                F.when(F.col("is_unusual"), 
                    F.element_at(F.array([F.lit(0), F.lit(1), F.lit(2), F.lit(3), F.lit(4), F.lit(23)]), (F.rand(3002) * 6 + 1).cast("int"))
                ).otherwise(
                    (F.rand(3003) * 17 + 6).cast("int")  # Hours 6-22
                )
            ) \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + (F.col("hour") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.element_at(F.array([F.lit(5000.0), F.lit(10000.0), F.lit(15000.0), F.lit(20000.0)]), (F.rand() * 4 + 1).cast("int"))) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", (F.rand() * 2 + 1).cast("int")) \
            .withColumn("location_changed_flag", F.lit(False))
    
    elif fraud_type == 'Amount Anomaly':
        # NOISE: Normal distribution centered at 45K with stddev 8K (was uniform 40-50K)
        # This creates overlap with legitimate high-value transactions
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 17 + 6).cast("int") * 3600) + ((F.rand() * 60).cast("int") * 60)).cast("timestamp")) \
            .withColumn("amount_inr", F.round(F.lit(45000) + F.randn(4001) * 8000, 2)) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", F.lit(0)) \
            .withColumn("location_changed_flag", F.lit(False))
    
    elif fraud_type == 'Temporal Anomaly':
        # NOISE: ~70% at unusual hours [0-5,23], ~30% at normal hours [6-22]
        fraud_txns = fraud_txns \
            .withColumn("is_unusual", F.rand(5001) < 0.70) \
            .withColumn("hour", 
                F.when(F.col("is_unusual"), 
                    F.element_at(F.array([F.lit(0), F.lit(1), F.lit(2), F.lit(3), F.lit(4), F.lit(5), F.lit(23)]), (F.rand(5002) * 7 + 1).cast("int"))
                ).otherwise(
                    (F.rand(5003) * 17 + 6).cast("int")  # Hours 6-22
                )
            ) \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + (F.col("hour") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.round(F.lit(15000) + F.rand() * 20000, 2)) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", F.lit(0)) \
            .withColumn("location_changed_flag", F.lit(False))
    
    elif fraud_type == 'Merchant Fraud':
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 24).cast("int") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.round(F.lit(10000) + F.rand() * 20000, 2)) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", F.lit(0)) \
            .withColumn("location_changed_flag", F.lit(False))
    
    elif fraud_type == 'Failed-Then-Success':
        fraud_txns = fraud_txns \
            .withColumn("timestamp", F.from_unixtime(F.lit(base_timestamp) + (F.rand() * date_range * 86400).cast("long") + ((F.rand() * 24).cast("int") * 3600)).cast("timestamp")) \
            .withColumn("amount_inr", F.element_at(F.array([F.lit(5000.0), F.lit(10000.0), F.lit(15000.0), F.lit(20000.0)]), (F.rand() * 4 + 1).cast("int"))) \
            .withColumn("device_changed_flag", F.lit(False)) \
            .withColumn("sim_change_recent", F.lit(False)) \
            .withColumn("mpin_attempts", F.lit(0)) \
            .withColumn("location_changed_flag", F.lit(False))
    
    # Common fraud fields
    fraud_txns = fraud_txns \
        .withColumn("txn_type", F.element_at(F.array([F.lit(t) for t in TXN_TYPES]), (F.rand() * len(TXN_TYPES) + 1).cast("int"))) \
        .withColumn("sender_id", F.col("user_id")) \
        .withColumn("sender_vpa", F.concat(
            F.element_at(F.array([F.lit(n) for n in ['alice', 'bob', 'charlie']]), (F.rand() * 3 + 1).cast("int")),
            (F.hash(F.col("user_id")) % 1000).cast("string"), F.lit("@"), F.lower(F.col("bank"))
        )) \
        .withColumn("sender_bank", F.col("bank")) \
        .withColumn("sender_state", F.col("state")) \
        .withColumn("sender_age_group", F.col("age_group")) \
        .withColumn("network_type", F.element_at(F.array([F.lit(n) for n in NETWORK_TYPES]), (F.rand() * len(NETWORK_TYPES) + 1).cast("int"))) \
        .withColumn("txn_status", F.lit("SUCCESS")) \
        .withColumn("is_fraud", F.lit(1)) \
        .withColumn("fraud_type", F.lit(fraud_type)) \
        .withColumn("fraud_type_id", F.lit(fraud_type_id))
    
    # Handle receiver (P2M vs P2P) - Merchant Fraud always P2M
    if fraud_type == 'Merchant Fraud':
        high_risk = merchant_spark_df.filter(F.col("is_high_risk") == True)
        merchant_for_fraud = high_risk if high_risk.count() > 0 else merchant_spark_df
        merchant_for_fraud = merchant_for_fraud.withColumn("merchant_idx", F.monotonically_increasing_id() % merchant_for_fraud.count())
        
        fraud_txns = fraud_txns \
            .withColumn("merchant_idx", (F.rand(fraud_type_id * 200) * merchant_for_fraud.count()).cast("int")) \
            .join(F.broadcast(merchant_for_fraud), "merchant_idx", "left") \
            .withColumn("receiver_id", F.col("merchant_id")) \
            .withColumn("receiver_vpa", F.col("merchant_vpa")) \
            .withColumn("receiver_bank", F.element_at(F.array([F.lit(b) for b in BANKS]), (F.rand() * len(BANKS) + 1).cast("int"))) \
            .withColumn("receiver_state", F.element_at(F.array([F.lit(s) for s in STATES]), (F.rand() * len(STATES) + 1).cast("int"))) \
            .withColumn("receiver_age_group", F.lit("NA")) \
            .withColumn("merchant_category", F.col("category")) \
            .withColumn("high_risk_merchant", F.lit(True))
    else:
        # Mix of P2M and P2P
        fraud_p2m = fraud_txns.filter(F.col("txn_type") == "P2M") \
            .withColumn("merchant_idx", (F.rand(fraud_type_id * 300) * NUM_MERCHANTS).cast("int"))
        
        fraud_p2m = fraud_p2m.join(F.broadcast(merchant_indexed), "merchant_idx", "left") \
            .withColumn("receiver_id", F.col("merchant_id")) \
            .withColumn("receiver_vpa", F.col("merchant_vpa")) \
            .withColumn("receiver_bank", F.element_at(F.array([F.lit(b) for b in BANKS]), (F.rand() * len(BANKS) + 1).cast("int"))) \
            .withColumn("receiver_state", F.element_at(F.array([F.lit(s) for s in STATES]), (F.rand() * len(STATES) + 1).cast("int"))) \
            .withColumn("receiver_age_group", F.lit("NA")) \
            .withColumn("merchant_category", F.col("category")) \
            .withColumn("high_risk_merchant", F.col("is_high_risk"))
        
        fraud_p2p = fraud_txns.filter(F.col("txn_type") != "P2M") \
            .withColumn("receiver_idx", (F.rand(fraud_type_id * 400) * NUM_USERS).cast("int"))
        
        fraud_p2p = fraud_p2p.join(F.broadcast(receiver_indexed), "receiver_idx", "left") \
            .withColumn("receiver_vpa", F.concat(
                F.element_at(F.array([F.lit(n) for n in ['alice', 'bob']]), (F.rand() * 2 + 1).cast("int")),
                (F.hash(F.col("receiver_id")) % 1000).cast("string"), F.lit("@"), F.lower(F.col("receiver_bank"))
            )) \
            .withColumn("merchant_category", F.lit(None).cast("string")) \
            .withColumn("high_risk_merchant", F.lit(False))
        
        fraud_txns = fraud_p2m.unionByName(fraud_p2p, allowMissingColumns=True)
    
    # Select final columns
    fraud_txns = fraud_txns.select(*final_columns)
    fraud_dfs.append(fraud_txns)
    fraud_type_id += 1

# Union all fraud transactions
fraud_final = fraud_dfs[0]
for df in fraud_dfs[1:]:
    fraud_final = fraud_final.unionByName(df, allowMissingColumns=True)

print(f"\n✓ Total fraud transactions generated: {fraud_final.count():,}")
print("✓ Realistic noise added: NO fraud type has 100% deterministic patterns")

In [0]:
# Union legitimate and fraud transactions
print("📊 Combining all transactions and sorting...")
df = legit_final.unionByName(fraud_final, allowMissingColumns=True)
df = df.orderBy("timestamp")

# Add temporal features
print("📊 Adding temporal features...")
df = df \
    .withColumn("hour_of_day", F.hour("timestamp")) \
    .withColumn("day_of_week", F.date_format("timestamp", "EEEE")) \
    .withColumn("is_weekend", F.col("day_of_week").isin(['Saturday', 'Sunday'])) \
    .withColumn("is_odd_hours", F.col("hour_of_day").isin([23, 0, 1, 2, 3, 4, 5]))

# Add derived features
print("📊 Adding derived features...")
df = df \
    .withColumn("amount_is_round", 
                (F.col("amount_inr") % 1000 == 0) | (F.col("amount_inr") % 5000 == 0))

print(f"✓ Final Spark DataFrame created with {df.count():,} rows and {len(df.columns)} columns")
df.show(5)

In [0]:
# Get current username dynamically for portability
try:
    username = spark.sql("SELECT current_user()").collect()[0][0]
    print(f"Current user: {username}")
except:
    username = os.environ.get('USER', 'default_user')
    print(f"Using username from environment: {username}")

# Construct portable output path
output_path = f'/Workspace/Users/{username}/Suraksha/data/synthetic_upi_txns.csv'
print(f"\n💾 Saving to {output_path}...")

# Ensure directory exists
output_dir = os.path.dirname(output_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✓ Created directory: {output_dir}")

# Convert Spark DataFrame to Pandas for CSV export
print("Converting Spark DataFrame to Pandas...")
pandas_df = df.toPandas()

# Save CSV
pandas_df.to_csv(output_path, index=False)
print(f"✓ File saved successfully!")
print(f"  File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

In [0]:
# Summary statistics
print("=" * 60)
print("✅ DATA GENERATION COMPLETE!")
print("=" * 60)

print(f"\n📊 Summary Statistics:")
total_count = df.count()
date_range_result = df.agg(F.min("timestamp").alias("min_ts"), F.max("timestamp").alias("max_ts")).collect()[0]
unique_senders = df.select("sender_id").distinct().count()
unique_receivers = df.select("receiver_id").distinct().count()

print(f"  Total transactions: {total_count:,}")
print(f"  Date range: {date_range_result['min_ts']} to {date_range_result['max_ts']}")
print(f"  Unique users: {unique_senders:,}")
print(f"  Unique receivers: {unique_receivers:,}")

print(f"\n🎯 Fraud Distribution:")
df.groupBy("fraud_type").count().orderBy(F.desc("count")).show(truncate=False)

print(f"\n💰 Amount Statistics:")
df.select("amount_inr").describe().show()

print(f"\n✅ Ready for ingestion into Delta Lake!")
print("   Next: Run advanced_solution/02_delta_ingestion notebook")